# 🏗️ Notebook 1: Google Calendar — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A calendar service like Google Calendar. Users:

- create **events** (one-off or recurring — "every Monday"),
- invite **guests** who can RSVP (yes/no/maybe),
- book **meeting rooms** as resources,
- share calendars with colleagues (read-only or read/write),
- get **reminders** by push or email.

Two parts are genuinely hard and deserve their own notebooks:

1. **Recurring events** — one "every weekday at 9am forever" event should NOT create infinite rows.
2. **Timezones** — storing "3pm" is ambiguous. Is that Tokyo 3pm or New York 3pm? And does daylight-saving time shift it?


## Functional requirements

| Area | What the user can do |
|---|---|
| Events | Create, edit, delete one-off or recurring events |
| Invites | Invite guests, see their RSVP (yes/no/maybe) |
| Rooms | Book a meeting room (a "resource" with its own calendar) |
| Availability | "Find me 30 min when Alice, Bob, and Room 7 are all free" |
| Sharing | Share my calendar: free/busy only, read, or read/write |
| Reminders | Get a push/email 10 minutes before an event |

## Non-functional requirements

- **Correctness of time**: handle timezones and daylight-saving transitions.
- **Low read latency**: opening the day view should feel instant (<200 ms).
- **Durable reminders**: a reminder must fire even if a server restarts.
- **Eventual consistency** across regions is acceptable for calendar views; invitations and RSVPs should converge within seconds.


## Back-of-envelope sizing

Rough numbers — the point is to reason about *orders of magnitude*, not to be exact.

- **Users**: 1 B active.
- **Events per user**: ~20 per week ⇒ ~1 T events over 10 years. Most are small (<1 KB).
- **Reads vs writes**: reads dominate heavily — every app open fetches a week. Expect read:write ≈ 100:1.
- **Reminders**: at peak (top of the hour, Monday 9am Americas) the scheduler may fire ~100k reminders/second — this is why reminders get their own durable queue (see the `reminder-alert` lab).


## High-level architecture

```
                     ┌──────────┐
                     │  Client  │  (web / mobile)
                     └────┬─────┘
                          ▼
                  ┌───────────────┐
                  │  API Gateway  │  auth, rate-limit, routing
                  └───┬─────┬─────┘
          ┌───────────┘     └────────────┐
          ▼                              ▼
   ┌──────────────┐               ┌─────────────────┐
   │ Event Svc    │               │ Sharing/ACL Svc │
   │ (CRUD, RRULE)│               │ (who sees what) │
   └──────┬───────┘               └────────┬────────┘
          │                                │
          ▼                                ▼
       Postgres                         Postgres
       (events,                         (acls)
        rrules,
        exceptions)

   ┌──────────────┐          ┌─────────────────────┐
   │ Invite Svc   │          │ Reminder Scheduler  │
   │ (RSVPs)      │          │ (durable timer q)   │
   └──────┬───────┘          └──────────┬──────────┘
          │                             │
          ▼                             ▼
       Postgres                  Push/Email workers
```

### Why split into services?

- **Event service** owns event rows and recurrence rules. For recurring events we store the *rule*, not every future instance.
- **Sharing/ACL service** answers "can Bob see Alice's calendar?" — a hot path on every read.
- **Invite service** tracks RSVPs separately so invitation fan-out doesn't hold up event writes.
- **Reminder scheduler** is a classic durable timer queue. Firing 100k reminders/s at peak is a different problem than CRUD, so it runs on its own infrastructure. See `06-system-designs/reminder-alert`.


## The two ideas that define the whole design

Read these twice — everything else follows:

1. **Store the rule, not the occurrences.** An event that repeats every Monday for 10 years is **one row** with `rrule='FREQ=WEEKLY;BYDAY=MO'`, *not* 520 rows. When the client asks "what's in May 2026?" we *expand* the rule inside that window. This keeps storage small and updates cheap.

2. **UTC is the source of truth.** Store `starts_at` in UTC, store the user's IANA timezone (e.g. `America/Los_Angeles`) as a separate field, and convert only for display. "3pm in LA" is ambiguous around DST; "2026-03-08T15:00:00Z rendered in America/Los_Angeles" is not.

We'll make both concrete with running code in notebooks 2 and 3.


## What each of the next notebooks covers

- **Notebook 2 — Data Model & APIs**: Pydantic models for Event/Invitation/Room, HTTP endpoints, and a bad→best walkthrough of timezone storage (the most common beginner mistake).
- **Notebook 3 — Deep Dives**: recurrence expansion with real `dateutil.rrule`, overriding or cancelling a single occurrence, and a bad→best **free/busy** algorithm (naive O(n·m) vs. sweep-line merge).
